<a href="https://colab.research.google.com/github/Mutasar/Proyek_Analisis_Sentimen/blob/main/Analisa_Sentimen_Ulasan_Aplikasi_Tokopedia_di_Playstore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Library

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from google.colab import drive
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.cluster import KMeans
from yellowbrick.cluster import KElbowVisualizer
import joblib
from google.colab import files


# 1. inisialisasi dataset

In [ ]:
# inisialisasi dataset
file_id = '1GLyhzyN3XNgmC3ftv4Klmvtlt_Dc7XzD'
download_url = f'https://drive.google.com/uc?id={file_id}'

# Membaca CSV
df = pd.read_csv(download_url)

In [ ]:
df.head()

,userName,score,at,content
0,Pengguna Google,5,2024-09-08 03:31:50,makasih toped
1,Pengguna Google,1,2024-09-08 03:29:56,Aplikasi php sudah banyak dikasih promo &sudah...
2,Pengguna Google,5,2024-09-08 03:26:38,mantab... 👍
3,Pengguna Google,5,2024-09-08 03:25:14,Good good
4,Pengguna Google,1,2024-09-08 03:24:01,Sangat buruk sebagai pengguna lama akun affali...


In [ ]:
# Tinjau jumlah baris kolom dan jenis data dalam dataset dengan info.

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 4 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   userName  500000 non-null  object
 1   score     500000 non-null  int64 
 2   at        500000 non-null  object
 3   content   499999 non-null  object
dtypes: int64(1), object(3)
memory usage: 15.3+ MB


In [ ]:
# Menampilkan statistik deskriptif dataset dengan menjalankan describe

df.describe()

,score
count,500000.000000
mean,4.074004
std,1.545313
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


In [ ]:
df = df.rename(columns={
    'userName': 'nama',
    'score': 'rating',
    'at': 'waktu',
    'content': 'ulasan'
})
print(df.columns)

Index(['nama', 'rating', 'waktu', 'ulasan'], dtype='object')


In [ ]:
df.head()

,nama,rating,waktu,ulasan
0,Pengguna Google,5,2024-09-08 03:31:50,makasih toped
1,Pengguna Google,1,2024-09-08 03:29:56,Aplikasi php sudah banyak dikasih promo &sudah...
2,Pengguna Google,5,2024-09-08 03:26:38,mantab... 👍
3,Pengguna Google,5,2024-09-08 03:25:14,Good good
4,Pengguna Google,1,2024-09-08 03:24:01,Sangat buruk sebagai pengguna lama akun affali...


# 2. Pembersihan Data

In [ ]:
# Mengecek dataset menggunakan isnull().sum()

print("Cek nilai null:\n", df.isnull().sum())

Cek nilai null:
 nama      0
rating    0
waktu     0
ulasan    1
dtype: int64


In [ ]:
# Mengecek dataset menggunakan duplicated().sum()

print("\nJumlah duplikasi:", df.duplicated().sum())


Jumlah duplikasi: 10


In [ ]:
#menghilangkan data duplikat
df = df.drop_duplicates()

In [ ]:
#cek hasil drop duplikat
print("\nJumlah duplikasi:", df.duplicated().sum())


Jumlah duplikasi: 0


# 3. Pembersihan text

In [37]:
# Fungsi pembersihan teks
def clean_text(text):
    if pd.isnull(text):  # Tangani NaN
        return ""
    text = text.lower()  # Ubah ke huruf kecil
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)  # Hapus URL
    text = re.sub(r'[^a-z\s]', '', text)  # Hapus karakter non-abjad
    tokens = word_tokenize(text)  # Tokenisasi
    stop_words = set(stopwords.words('indonesian'))  # Ubah ke bahasa Indonesia jika perlu
    tokens = [word for word in tokens if word not in stop_words]  # Hapus stopwords
    return ' '.join(tokens)

# Terapkan ke kolom teks (misalnya 'ulasan')
df['ulasan'] = df['ulasan'].apply(clean_text)

In [ ]:
# Tampilkan beberapa baris pertama dari kolom 'ulasan' setelah pembersihan
print("\nHasil pembersihan teks:")
print(df['ulasan'].head())


Hasil pembersihan teks:
0                                        makasih toped
1    aplikasi php dikasih promo dibayar diklik diba...
2                                               mantab
3                                            good good
4    buruk pengguna akun affalite blokir rugikan share
Name: ulasan, dtype: object


In [34]:
# Buat dataframe baru untuk menampung data yang sudah dilakukan preprocessing
clean_df = df.copy()

clean_df['ulasan'] = clean_df['ulasan'].str.lower()

In [36]:
print(f"Before: '{df['ulasan'][4]}'")
print(f"After: '{clean_df['ulasan'][4]}'")

Before: 'Sangat buruk sebagai pengguna lama akun affalite di blokir tanpa sebab amat sangat di rugikan dalam hal share'
After: 'sangat buruk sebagai pengguna lama akun affalite di blokir tanpa sebab amat sangat di rugikan dalam hal share'


# Labeling

Pembagian data menjadi data sentimen berlabel positif dan negatif dengan angka 1 untuk positif dan angka 0 untuk negatif. Pengklasifikasian ini dilakukan pada ulasan yang memiliki rating 4 dan 5 sebagai sentimen positif dan rating 3 sampai 1 sebagai sentimen negatif.

In [ ]:
# sentimen berdasarkan rating
def get_sentiment_label(rating):
    if rating >= 4:
        return 1  # Positif
    elif rating <= 3:
        return 0  # Negatif
    else:
        return None  # Jika rating tidak valid

# menerapkan fungsi ke DataFrame
df['label'] = df['rating'].apply(get_sentiment_label)

print(df.head())


              nama  rating                waktu  \
0  Pengguna Google       5  2024-09-08 03:31:50   
1  Pengguna Google       1  2024-09-08 03:29:56   
2  Pengguna Google       5  2024-09-08 03:26:38   
3  Pengguna Google       5  2024-09-08 03:25:14   
4  Pengguna Google       1  2024-09-08 03:24:01   

                                              ulasan  label  label_sentimen  
0                                      makasih toped      1               1  
1  aplikasi php dikasih promo dibayar diklik diba...      0               0  
2                                             mantab      1               1  
3                                          good good      1               1  
4  buruk pengguna akun affalite blokir rugikan share      0               0  


In [ ]:
df["label"].value_counts()

,count
label,
1,379625
0,120365


# Menghitung Kata Dengan TF-IDF

In [ ]:
#untuk menghitung jumlah kata yang telah di steming
cv = CountVectorizer()
term_fit = cv.fit(Ulasan)

print (len(term_fit.vocabulary_))

121513


In [ ]:
term_fit.vocabulary_ #mengurutkan berdasarkan urutab abjad kata

{'makasih': 61831,
 'toped': 113354,
 'aplikasi': 5152,
 'php': 87102,
 'dikasih': 26525,
 'promo': 89696,
 'dibayar': 25258,
 'diklik': 26799,
 'dibatalkan': 25126,
 'sistem': 102218,
 'tolong': 113148,
 'perbaiki': 85739,
 'kmu': 53918,
 'tokopedia': 111536,
 'mantab': 63045,
 'good': 37575,
 'buruk': 18788,
 'pengguna': 84581,
 'akun': 2235,
 'affalite': 1064,
 'blokir': 16442,
 'rugikan': 94112,
 'share': 100700,
 'verifikasi': 117433,
 'nomor': 75960,
 'hp': 41998,
 'gabisa': 35049,
 'dipake': 27516,
 'udah': 115648,
 'tau': 106795,
 'ribet': 93165,
 'beli': 11886,
 'tuh': 115054,
 'payah': 81790,
 'diskonnya': 28894,
 'gokil': 37497,
 'oke': 77980,
 'hallo': 39937,
 'tokopediatolong': 112331,
 'dipermudah': 27784,
 'pembayaran': 83313,
 'fitur': 34447,
 'coddikarenakan': 21492,
 'merepotkan': 69613,
 'membantu': 66002,
 'job': 47026,
 'tq': 113842,
 'yaa': 119890,
 'promonya': 89849,
 'gak': 35421,
 'mempermudah': 66757,
 'voucher': 117835,
 'indomaret': 43364,
 'ket': 52666,
 't

In [ ]:
#kolom pertama ini berarti jumlah dokumen
#kolom kedua berarti letak katanya
#kolom ketiga hasil dari tf

term_frequency_all = term_fit.transform(Ulasan)
print (term_frequency_all)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 2336290 stored elements and shape (499990, 121513)>
  Coords	Values
  (0, 61831)	1
  (0, 113354)	1
  (1, 5152)	1
  (1, 25126)	1
  (1, 25258)	1
  (1, 26525)	1
  (1, 26799)	1
  (1, 53918)	1
  (1, 85739)	1
  (1, 87102)	1
  (1, 89696)	1
  (1, 102218)	2
  (1, 111536)	1
  (1, 113148)	1
  (2, 63045)	1
  (3, 37575)	2
  (4, 1064)	1
  (4, 2235)	1
  (4, 16442)	1
  (4, 18788)	1
  (4, 84581)	1
  (4, 94112)	1
  (4, 100700)	1
  (5, 2235)	1
  (5, 5152)	1
  :	:
  (499982, 60835)	1
  (499982, 71950)	1
  (499982, 76477)	1
  (499982, 88878)	1
  (499982, 99121)	1
  (499982, 109906)	1
  (499983, 2681)	1
  (499984, 7338)	1
  (499984, 44174)	1
  (499984, 49109)	1
  (499984, 66270)	1
  (499984, 82834)	1
  (499984, 84266)	1
  (499984, 88910)	1
  (499985, 7338)	1
  (499986, 5152)	1
  (499986, 62506)	1
  (499986, 66440)	1
  (499986, 71950)	1
  (499986, 83125)	1
  (499986, 116915)	2
  (499986, 119017)	1
  (499987, 66002)	1
  (499988, 66870)	1
  (499989, 1

In [ ]:
ulasan_tf = Ulasan[1] #memanggil kata pada index ke 1
print (ulasan_tf)

aplikasi php dikasih promo dibayar diklik dibatalkan sistem tolong perbaiki sistem kmu tokopedia


In [ ]:
term_frequency = term_fit.transform([ulasan_tf]) #hanya menampilkan hasil document 1
print (term_frequency)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 12 stored elements and shape (1, 121513)>
  Coords	Values
  (0, 5152)	1
  (0, 25126)	1
  (0, 25258)	1
  (0, 26525)	1
  (0, 26799)	1
  (0, 53918)	1
  (0, 85739)	1
  (0, 87102)	1
  (0, 89696)	1
  (0, 102218)	2
  (0, 111536)	1
  (0, 113148)	1


In [ ]:
dokumen = term_fit.transform(Ulasan) #hasil perhitungan tf idf dalam 1 doc
tfidf_transformer = TfidfTransformer().fit(dokumen)
print (tfidf_transformer.idf_)

tfidf=tfidf_transformer.transform(term_frequency)
print (tfidf) #hasil manual dengan sistem pyhton

[10.19051974 10.48475922 10.51142746 ... 13.02373309 13.4291982
 13.4291982 ]
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 12 stored elements and shape (1, 121513)>
  Coords	Values
  (0, 5152)	0.1500877561331256
  (0, 25126)	0.21854963180161935
  (0, 25258)	0.29542305587653694
  (0, 26525)	0.29105483550354017
  (0, 26799)	0.3806897990703465
  (0, 53918)	0.421111174982977
  (0, 85739)	0.25836161049864026
  (0, 87102)	0.30800721082850896
  (0, 89696)	0.18172532994625934
  (0, 102218)	0.43044778204295603
  (0, 111536)	0.12333918570290311
  (0, 113148)	0.20080232277458962


# menggunakan Algoritma Deep Learning

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 499990 entries, 0 to 499999
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   nama            499990 non-null  object
 1   rating          499990 non-null  int64 
 2   waktu           499990 non-null  object
 3   ulasan          499990 non-null  object
 4   label           499990 non-null  int64 
 5   label_sentimen  499990 non-null  int64 
dtypes: int64(3), object(3)
memory usage: 42.8+ MB


In [ ]:
# Menyiapkan Data Train dan Test (Keep this section as it splits the data)
data_label['Ulasan_clean'] = data_label['Ulasan_clean'].fillna("tidak ada komentar")

from sklearn.model_selection import train_test_split

X = data_label['Ulasan_clean']
y = data_label['label']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.1, stratify=y, random_state=30)



In [ ]:
# Parameters for tokenization and padding
vocab_size = 10000  # Adjust based on your vocabulary size
embedding_dim = 16
max_length = 100    # Adjust based on the average length of your reviews
trunc_type = 'post'
padding_type = 'post'
oov_tok = "<OOV>"


In [ ]:
# Tokenize the training data
tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(X_train)

In [ ]:
# Convert text to sequences
train_sequences = tokenizer.texts_to_sequences(X_train)
test_sequences = tokenizer.texts_to_sequences(X_test)


In [ ]:
# Pad sequences to ensure consistent length
train_padded = pad_sequences(train_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)
test_padded = pad_sequences(test_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

In [ ]:
# Convert labels to numpy arrays (required for TensorFlow)
y_train = np.array(y_train)
y_test = np.array(y_test)

In [ ]:
# Build the LSTM model
model = Sequential([
    # Remove input_length as it is deprecated
    Embedding(vocab_size, embedding_dim),
    LSTM(32), # You can adjust the number of units
    Dropout(0.5), # Add dropout for regularization
    Dense(1, activation='sigmoid') # Sigmoid for binary classification
])

In [ ]:
# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])


In [ ]:
# Print the model summary
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train the model
num_epochs = 10 # You can adjust the number of epochs
history = model.fit(train_padded, y_train, epochs=num_epochs, validation_data=(test_padded, y_test), verbose=2)

Epoch 1/10


In [ ]:
# Evaluate the model
loss, accuracy = model.evaluate(test_padded, y_test, verbose=0)
print(f'Test Loss: {loss:.4f}')
print(f'Test Accuracy: {accuracy:.4f}')

In [ ]:
# Make predictions
y_pred_dl = model.predict(test_padded)
y_pred_dl = (y_pred_dl > 0.5).astype(int) # Convert probabilities to class labels

In [ ]:
# Display confusion matrix and classification report for Deep Learning model

print('\n--------------------- Deep Learning Confusion Matrix  ----------------------------')
print(confusion_matrix(y_test, y_pred_dl))
print('--------------------- Deep Learning Classification Report  ----------------------------')
print(classification_report(y_test, y_pred_dl))

In [ ]:

# Optional: Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()

plt.show()

In [ ]:
data_label.to_excel("data_label.xlsx")

In [ ]:
sentimen_data=pd.value_counts(data_label["label"], sort= True)
sentimen_data.plot(kind= 'bar', color= ["green", "red"])
plt.title('Bar chart')
plt.show()

Dapat dilihat bahwa isi ulasan produk lebih banyak pada label sentimen 1 atau ulasan dengan rating postitif ini berarti pelanggan yang menggunakan marketplace Tokopedia dan melakukan transaksi pembelian pada produk masker Kesehatan merasa puas bertansaksi di marketplace Tokopedia dan prosuk masker Kesehatan sehingga memberikan feedback atau ulasan komentar lebih banyak yang positif.

In [ ]:
from wordcloud import WordCloud

**Ulasan Negatif**

In [ ]:
train_s0 = data_label[data_label["label"] == 0]

In [ ]:
train_s0["Ulasan_clean"] = train_s0["Ulasan_clean"].fillna("tidak ada komentar")

In [ ]:
train_s0

In [ ]:
all_text_s0 = ' '.join(word for word in train_s0["Ulasan_clean"])
wordcloud = WordCloud(colormap='Reds', width=1000, height=1000, mode='RGBA', background_color='white').generate(all_text_s0)
plt.figure(figsize=(20,10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.margins(x=0, y=0)
plt.show()

Dari visualisasi diatas merupakan wordcloud kata yang paling banyak muncul pada isi ulasan yang memiliki label sentimen negatif. Kata yang paling sering muncul dan mengarah ke ulasan negatif membahas seputar : barang, harga, kotak, penyok, box, dus, tipis, putus, sobek, kualitas, karet, bolong dan sebagainya. Sehingga dari kata-kata ini bisa menjadi masukan untuk penjual dan marketplace Tokopedia untuk meningkatkan kualitas barang (produk masker kesehatan), harga, kualitas pengiriman atau pengemasan, serta kualitas produk masker Kesehatan yang paling banyak disebutkan pelanggan dalam hasil Analisa ulasan sentimen yang negatif.

**Ulasan Positif**

In [ ]:
train_s1 = data_label[data_label["label"] == 1]

In [ ]:
train_s1["Ulasan_clean"] = train_s1["Ulasan_clean"].fillna("tidak ada komentar")

In [ ]:
train_s1

In [ ]:
all_text_s1 = ' '.join(word for word in train_s1["Ulasan_clean"])
wordcloud = WordCloud(colormap='Blues', width=1000, height=1000, mode='RGBA', background_color='white').generate(all_text_s1)
plt.figure(figsize=(20,10))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.title("Ulasan Positif")
plt.margins(x=0, y=0)
plt.show()

Dari visualisasi diatas merupakan wordcloud kata yang paling banyak muncul pada ulasan yang memiliki label sentimen positif. Kata yang paling sering muncul dan mengarah ke ulasan positif membahas seputar : barang, cepat, bagus, masker, aman, kualitas, sesuai, rapi, respon, aman, recommended, dan sebagainya. Sehingga dari kata-kata ini bisa menjadi masukan untuk penjual dan marketplace Tokopedia untuk menjaga kualitas atau meningkatkan kembali kualitas barang (produk masker kesehatan), kualitas yang sesuai dan aman, serta respon penjual paling banyak disebutkan pelanggan dalam hasil Analisa ulasan sentimen yang positif.

# Menyiapkan Data Train dan Test

Pada proses ini kami menggunakan library sklearn.model_selection dengan modul train_test _split untuk membagi data latih (X_train dan y_train) dan data uji (X_test dan y_test) dengan persentasi data latih 70% dan data uji 30% serta memilih label data yaitu yang merupakan variable independen dari data kami yaitu kolom label untuk dijadikan parameter klasifikasi prediksi.

In [ ]:
data_label['Ulasan_clean'] = data_label['Ulasan_clean'].fillna("tidak ada komentar")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data_label['Ulasan_clean'], data_label['label'],
                                                    test_size=0.1, stratify=data_label['label'], random_state=30)

# TF-IDF

Pada proses ini kami menggunakan pembobotan TF-IDF(term frequency–inverse document) untuk menghitung manual dengan menggunakan python pembobotan kata dalam dokumen data ulasan.

In [ ]:
import numpy as np

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(decode_error='replace', encoding='utf-8')

In [ ]:
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

print(X_train.shape)
print(X_test.shape)

In [ ]:
X_train = X_train.toarray()

In [ ]:
X_test = X_test.toarray()

# Machine Learning

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

#deklarasi metode cross validation
cv_method = RepeatedStratifiedKFold(n_splits=5,  n_repeats=3, random_state=999)
#tuning hyperparameter menggunakan gridsearch

params_NB = {'var_smoothing': np.logspace(0,-9, num=100)}
gscv_nb = GridSearchCV(estimator=nb,
                 param_grid=params_NB,
                 cv=cv_method,   # use any cross validation technique
                 verbose=1,
                 scoring='accuracy')

#Fitting ke Model
gscv_nb.fit(X_train,y_train)
#mendapatkan hyperparameters terbaik
gscv_nb.best_params_

In [ ]:
nb = GaussianNB(var_smoothing=1.0)

In [ ]:
nb.fit(X_train, y_train)

In [ ]:
y_pred_nb = nb.predict(X_test)

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [ ]:
print('--------------------- confusion matrix  ----------------------------')
print(confusion_matrix(y_test, y_pred_nb))
print('--------------------- classification report  ----------------------------')
print(classification_report(y_test, y_pred_nb))

Setelah dilakukan pembagian data latih dan data uji serta pembobotan tf-idf selanjutnya dapat dilakukan proses klasifikasi prediksi menggunakan model algoritma Naïve Bayes seperti proses yang ditunjukkan pada gambar 4. Didapatkan model algoritma Naïve Bayes dapat memberikan akurasi yang cukup baik sampari 88%.

> Dari hasil penelitian menggunakan Metode Algoritma Naïve Bayes untuk mengetahui sentimen ulasan pengguna dengan klasifikasi 2 kelas positif dan negative dengan pendekatan NLP menghasilkan nilai akurasi sebesar 88%. Selain itu, didapatkan bahwa Analisa Sentimen pada ulasan marketplace Tokopedia pada produk masker kesehatan menunjukan lebih banyak pada ulasan yang positif. Ini berarti pelayanan dan produk masker Kesehatan yang disediakan di marketplace Tokopedia sudah cukup baik.


> Dari hasil Analisis diatas dapat disimpulkan hasil scraping yang kami dapat dari produk pencarian masker kesehatan pada Tokopedia menampilkan data yang menunjukan ulasan positif lebih dominan daripada hasil ulasan negatif dan untuk ulasan negatif kata yang paling sering muncul adalah seputar kualitas produk yang tipis,mudah putus, sobek atau bolong, kualitas karetnya dan  pada pengemasan yaitu kotak/dus penyok dan untuk analisa positif kata yang paling sering muncul adalah seputar kualitas produk yang sesuai dan rapi, pengiriman yang aman dan cepat dan respon penjual.